# 01 - Generate synthetic images

PRD sections 3.4-3.6 and 3.11, flow 2. Course tools: attributed data generation with an LLM (2262), Stable Diffusion
text-to-image / inpainting / GLIGEN (2263, 2225), InstructPix2Pix (2263), negative prompts and CFG (2422).

1. Real images and the split (needed as source images for method B)
2. Attributes from an LLM, reviewed by hand (the notebook stops until they are marked reviewed)
3. Method A: add an object, a lookalike or a forbidden object to a background; the box is saved automatically
4. Method B: change the capture conditions of real tune images; the boxes are carried over
5. Samples and counts

Every image gets a `meta/<id>.json` in Drive, written last. A rerun after a disconnect skips finished images.

In [ ]:
# Colab's preinstalled torchao is older than what diffusers/peft expect; we don't use it.
%pip uninstall -q -y torchao
import importlib.util, sys
if "torchao" in sys.modules:
    raise RuntimeError("An old torchao is still loaded from a previous run. "
                       "Runtime > Disconnect and delete runtime, then Run all again.")
if importlib.util.find_spec("torchao") is not None:
    raise RuntimeError(f"torchao could not be removed: {importlib.util.find_spec('torchao').origin}")

%pip install -q diffusers transformers peft accelerate openai

In [ ]:
BRANCH = "dev"  # set to your branch name to run a branch

import os, subprocess, sys, yaml
REPO = "/content/genai-shelter-validator"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "https://github.com/AlmogSchwartzberg/genai-shelter-validator.git", REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "-q"], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", BRANCH], check=True)
subprocess.run(["git", "-C", REPO, "pull", "-q", "origin", BRANCH], check=True)
cfg = yaml.safe_load(open(f"{REPO}/config.yaml"))
sys.path.insert(0, f"{REPO}/Code")

from google.colab import drive
drive.mount("/content/drive")
ROOT = cfg["drive_root"]
os.makedirs(ROOT, exist_ok=True)
print("Config loaded, Drive ready at", ROOT)

## 1. Load the objects table and background images

The table gives the classes (5 required, 3 lookalikes, 2 forbidden) and the size range of each object.
Real images are split 80/20 with seed 42 (10% of the 80% is the control split). Method B only uses the **tune** part,
so no test image ever feeds the generator.

In [ ]:
import gc, json, random, re, shutil, time
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw
from tqdm.auto import tqdm
import common as C

device = C.device
gen = cfg["generation"]
S = gen["image_size"]
dtype = torch.float16 if device.type == "cuda" else torch.float32
variant = "fp16" if device.type == "cuda" else None

table = C.load_objects(REPO, cfg)
classes = table["name"].tolist()
print(table[["name", "role", "lookalike_of", "size_range"]].to_string(index=False))

models = [gen["model"], gen["background_model"], gen["condition_model"]]
if gen["add_objects_backend"] == "gligen":
    models.append(gen["gligen_model"])
C.check_models(models)

if os.path.exists(f"{ROOT}/real/annotations.json"):
    real = C.split_real(C.load_real(ROOT, classes), cfg, ROOT)
    display(C.split_table(real, classes))
    C.check_test_requirements(real, table)
    duplicates = C.find_duplicates(real)
    if duplicates:
        print("WARNING: the same image appears in different splits:", duplicates)
    tune = [r for r in real if r["split"] == "tune"]
else:
    tune = []
    print("No real images in Drive yet (real/annotations.json). Method A works without them; method B needs them.")

## 2. Attributes from an LLM (PRD 3.11)

The LLM lists shelter types, lighting, clutter and several visual variants of every class (for a lookalike, every variant
shows its defect). Method A samples from these lists to build each prompt, as in the course dog-breed example.

**The notebook stops here until you review `synthetic/attributes.json` in Drive** and set `"reviewed": true`.
Then copy the file to `Data/attributes.json` in the repo, so a teammate can reuse it without an API key.

In [ ]:
attr_path = f"{ROOT}/synthetic/attributes.json"
repo_attr = f"{REPO}/Data/attributes.json"
os.makedirs(f"{ROOT}/synthetic", exist_ok=True)


def ask_llm_for_attributes():
    from google.colab import userdata
    from openai import OpenAI
    try:
        key = userdata.get(cfg["llm"]["secret_name"])
    except Exception:
        raise RuntimeError(f"No LLM key. Add a Colab secret named '{cfg['llm']['secret_name']}' "
                           "(key icon in the left bar) and switch on notebook access, then rerun.")
    rows = table[["name", "role", "lookalike_of", "appearance", "cause_or_context"]].to_dict("records")
    prompt = (
        "You help build a synthetic photo dataset of public shelters (protected rooms) for object detection.\n"
        "Return ONLY a JSON object with these keys:\n"
        '"shelter_types": 10 short descriptions of different shelter rooms,\n'
        '"lighting": 8 short lighting descriptions,\n'
        '"clutter": 6 short descriptions of how tidy or cluttered the room is,\n'
        f'"objects": an object whose keys are exactly these class names: {classes}. For every class give 12 different '
        "visual descriptions (at most 15 words each) of how it looks in a photo, based on the table below. "
        "For a class with role lookalike, every description must show the visible defect that makes it a lookalike.\n"
        f"Table: {json.dumps(rows)}")
    response = OpenAI(api_key=key).responses.create(model=cfg["llm"]["model"], input=prompt)
    text = re.sub(r"^```(?:json)?|```$", "", response.output_text.strip(), flags=re.M).strip()
    return json.loads(text)


if not os.path.exists(attr_path) and os.path.exists(repo_attr):
    shutil.copy(repo_attr, attr_path)
if not os.path.exists(attr_path):
    attrs = ask_llm_for_attributes()
    attrs.update({"reviewed": False, "llm": cfg["llm"]["model"]})
    json.dump(attrs, open(attr_path, "w"), indent=1)
    raise RuntimeError(f"attributes.json was written to {attr_path}. Review it, set \"reviewed\": true, then rerun.")

attrs = json.load(open(attr_path))
if attrs.get("reviewed") is not True:
    raise RuntimeError(f"{attr_path} is not marked reviewed. Check the lists, set \"reviewed\": true, then rerun.")
for key in ("shelter_types", "lighting", "clutter"):
    assert attrs.get(key), f"attributes.json has no '{key}'"
for name in classes:
    assert attrs["objects"].get(name), f"attributes.json has no variants for class '{name}'"
print({k: len(v) for k, v in attrs.items() if isinstance(v, list)}, "| variants per class:",
      {n: len(v) for n, v in attrs["objects"].items()})

## 3. Method A: add objects with Stable Diffusion inpainting (boxes saved automatically)

Per image: the LLM attributes build a prompt, text-to-image makes an empty shelter background (negative prompt removes
the target objects), a random box inside the class size range becomes the inpainting mask, and the object is painted into
it. The mask is the label: the box and the class name are saved with the image.
Small boxes use `padding_mask_crop`, so SD repaints a zoomed-in crop instead of 4 latent pixels.
`add_objects_backend: gligen` in the config switches to GLIGEN (box + phrase instead of a mask).
Items are created round-robin over the classes, so an interrupted run is still class balanced.

In [ ]:
from diffusers import AutoPipelineForText2Image, AutoPipelineForInpainting, StableDiffusionGLIGENPipeline

NEG_BACKGROUND = "people, text, watermark, blurry, fire extinguisher, gas cylinder, mirror, picture frame, sign, bottle"
NEG_OBJECT = "blurry, deformed, text, watermark, multiple objects, cropped, cartoon"
N_A = gen["images_per_class"]
assert N_A < 1000, "item seeds assume fewer than 1000 images per class"

text2image = AutoPipelineForText2Image.from_pretrained(
    gen["background_model"], torch_dtype=dtype, variant=variant, safety_checker=None, requires_safety_checker=False).to(device)
if gen["add_objects_backend"] == "gligen":
    painter = StableDiffusionGLIGENPipeline.from_pretrained(gen["gligen_model"], variant=variant, torch_dtype=dtype).to(device)
else:
    painter = AutoPipelineForInpainting.from_pretrained(
        gen["model"], torch_dtype=dtype, variant=variant, safety_checker=None, requires_safety_checker=False).to(device)


def sample_box(name, rng):
    """Random box inside the class size range (the range is the longer side, in pixels at 512)."""
    lo, hi = C.parse_size(table.loc[table["name"] == name, "size_range"].iloc[0])
    long_side = min(rng.randint(lo, hi), S - 1)
    short_side = max(8, int(long_side * rng.uniform(0.5, 1.0)))
    w, h = (long_side, short_side) if rng.random() < 0.5 else (short_side, long_side)
    x0, y0 = rng.randint(0, S - w), rng.randint(0, S - h)
    return [x0, y0, x0 + w, y0 + h]


def make_method_a(name, i):
    item_id = f"A_{name}_{i:03d}"
    seed = cfg["seed"] * 100_000 + classes.index(name) * 1000 + i
    rng = random.Random(seed)
    ctx = {k: rng.choice(attrs[k]) for k in ("shelter_types", "lighting", "clutter")}
    variant_index = rng.randrange(len(attrs["objects"][name]))
    description = attrs["objects"][name][variant_index]
    box = sample_box(name, rng)
    background_prompt = (f"photo of the inside of a {ctx['shelter_types']}, {ctx['lighting']}, {ctx['clutter']}, "
                         "empty walls and floor, realistic photograph")
    prompt = f"{description}, {ctx['lighting']}, inside a {ctx['shelter_types']}, high resolution photo"
    generator = torch.Generator(device.type).manual_seed(seed)
    start = time.time()
    background = text2image(prompt=background_prompt, negative_prompt=NEG_BACKGROUND, height=S, width=S,
                            num_inference_steps=gen["steps"], guidance_scale=gen["guidance_scale"], generator=generator).images[0]
    if gen["add_objects_backend"] == "gligen":
        image = painter(prompt=prompt, negative_prompt=NEG_OBJECT, gligen_phrases=[description],
                        gligen_inpaint_image=background, gligen_boxes=[[b / S for b in box]],
                        gligen_scheduled_sampling_beta=1, output_type="pil", num_inference_steps=gen["steps"],
                        guidance_scale=gen["guidance_scale"], generator=generator).images[0]
    else:
        mask = Image.new("L", (S, S), 0)
        ImageDraw.Draw(mask).rectangle(box, fill=255)
        small = max(box[2] - box[0], box[3] - box[1]) < 128
        image = painter(prompt=prompt, negative_prompt=NEG_OBJECT, image=background, mask_image=mask, height=S, width=S,
                        num_inference_steps=gen["steps"], guidance_scale=gen["guidance_scale"], generator=generator,
                        padding_mask_crop=32 if small else None).images[0]
    meta = {"id": item_id, "file": f"images/{item_id}.png", "method": "add_objects", "condition": "clean",
            "prompt": prompt, "negative_prompt": NEG_OBJECT, "background_prompt": background_prompt,
            "attributes": {**ctx, "object": name, "variant_index": variant_index, "variant": description,
                           "attributes_file": "synthetic/attributes.json"},
            "seed": seed, "model": gen["model"], "backend": gen["add_objects_backend"], "size": [S, S],
            "boxes": [{"name": name, "xyxy": box}], "source_id": None, "filter": None,
            "seconds": round(time.time() - start, 1)}
    C.save_item(ROOT, item_id, image, meta)


todo = [(name, i) for i in range(N_A) for name in classes if not C.item_done(ROOT, f"A_{name}_{i:03d}")]
print(f"Method A: {len(todo)} of {N_A * len(classes)} images left")
for name, i in tqdm(todo):
    make_method_a(name, i)

del text2image, painter
gc.collect()
torch.cuda.empty_cache()

## 4. Method B: change capture conditions (objects stay in place)

InstructPix2Pix edits a real tune image with a text instruction while keeping its layout (a high `image_guidance_scale`).
The real boxes are scaled to 512x512 and saved with the new image. Mode 5 of notebook 03 applies classical augmentation
to the same source images, so the comparison asks whether a generative model adds anything over simple image operations.
Check by eye that the objects did not move (PRD 3.5: 20 random samples).

In [ ]:
from diffusers import StableDiffusionInstructPix2PixPipeline, EulerAncestralDiscreteScheduler

CONDITION_PROMPTS = {
    "dark": "make the room very dark, lights switched off, only a little dim light",
    "side_light": "strong light coming from the {side} side through a window, long hard shadows",
    "glare": "strong bright glare and lens flare, overexposed highlights",
    "blur": "make the photo out of focus and blurry, camera shake",
    "noise": "make it a grainy low light photo with heavy sensor noise",
    "background": "change the color of the walls and the material of the floor, keep every object where it is",
}
NEG_CONDITION = "text, watermark, new objects, moved objects, deformed"
unknown = [c for c in gen["conditions"] if c not in CONDITION_PROMPTS]
assert not unknown, f"No instruction for conditions {unknown}; add them to CONDITION_PROMPTS"
N_B = gen["images_per_condition"]
assert N_B < 1000, "item seeds assume fewer than 1000 images per condition"

sources = [r for r in tune if r["boxes"]]
todo_b = [(c, j) for j in range(N_B) for c in gen["conditions"] if not C.item_done(ROOT, f"B_{c}_{j:03d}")]
if not sources:
    print("Method B skipped: no labeled real tune images yet.")
elif todo_b:
    ip2p = StableDiffusionInstructPix2PixPipeline.from_pretrained(
        gen["condition_model"], torch_dtype=dtype, safety_checker=None).to(device)
    ip2p.scheduler = EulerAncestralDiscreteScheduler.from_config(ip2p.scheduler.config)


def make_method_b(condition, j):
    item_id = f"B_{condition}_{j:03d}"
    seed = cfg["seed"] * 100_000 + 50_000 + gen["conditions"].index(condition) * 1000 + j
    rng = random.Random(seed)
    source = rng.choice(sources)
    instruction = CONDITION_PROMPTS[condition].format(side=rng.choice(["left", "right"]))
    image_in = Image.open(source["path"]).convert("RGB").resize((S, S))
    sx, sy = S / source["width"], S / source["height"]
    boxes = [{"name": classes[label], "xyxy": [x0 * sx, y0 * sy, x1 * sx, y1 * sy]}
             for (x0, y0, x1, y1), label in zip(source["boxes"], source["labels"])]
    generator = torch.Generator(device.type).manual_seed(seed)
    start = time.time()
    image = ip2p(prompt=instruction, image=image_in, negative_prompt=NEG_CONDITION, num_inference_steps=gen["steps"],
                 guidance_scale=gen["guidance_scale"], image_guidance_scale=1.6, generator=generator).images[0]
    meta = {"id": item_id, "file": f"images/{item_id}.png", "method": "capture_conditions", "condition": condition,
            "prompt": instruction, "negative_prompt": NEG_CONDITION, "attributes": {"condition": condition},
            "seed": seed, "model": gen["condition_model"], "backend": "instruct-pix2pix", "size": [S, S],
            "boxes": boxes, "source_id": source["id"], "source_url": source["source_url"], "filter": None,
            "seconds": round(time.time() - start, 1)}
    C.save_item(ROOT, item_id, image, meta)


if sources and todo_b:
    print(f"Method B: {len(todo_b)} of {N_B * len(gen['conditions'])} images left")
    for condition, j in tqdm(todo_b):
        make_method_b(condition, j)
    del ip2p
    gc.collect()
    torch.cuda.empty_cache()

## 5. Show 16 random samples and counts per object type

In [ ]:
import matplotlib.pyplot as plt

all_meta = C.read_meta(ROOT)
rows = [{"object": b["name"], "method": "A" if m["method"] == "add_objects" else "B"} for m in all_meta for b in m["boxes"]]
counts = pd.crosstab(pd.Categorical([r["object"] for r in rows], categories=classes), [r["method"] for r in rows])
counts.index.name = "object"
display(counts)
display(pd.Series([m["condition"] for m in all_meta]).value_counts().rename("images per condition").to_frame())

pick = random.Random(cfg["seed"]).sample(all_meta, min(16, len(all_meta)))
fig, axes = plt.subplots(4, 4, figsize=(14, 14))
for ax in axes.flat:
    ax.axis("off")
for ax, m in zip(axes.flat, pick):
    img = Image.open(f"{ROOT}/synthetic/{m['file']}").convert("RGB")
    for b in m["boxes"]:
        ImageDraw.Draw(img).rectangle(b["xyxy"], outline="red", width=3)
    ax.imshow(img)
    ax.set_title(f"{m['method'][:3]}/{m['condition']}: {', '.join(sorted({b['name'] for b in m['boxes']}))}", fontsize=8)
plt.tight_layout()
plt.show()